# Step 5 — Two-tower model (query + user tower, item tower)

Trains the candidate-conditioned tower model on GPU with cross-fitting and scores every displayed item. Because its user vector depends on each candidate, it scores the displayed slate rather than providing a single query vector for standard nearest-neighbour retrieval.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [1]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/harvestingmoon/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-main-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    # Reuse the same checkout throughout this runtime so stages use one revision.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if (ROOT / ".git").exists():
    print("Repository revision:", subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True).strip())
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

Repository revision: 1ab81f2
config configs/default.yaml | scope full | raw data /home/ubuntu/bt4222-two-tower-recsys/data/raw | outputs /home/ubuntu/bt4222-two-tower-recsys/outputs/full


## 5.1 Architecture (`src/jds/twotower/model.py`)

* **Item tower:** mean term embeddings of title (20 terms), brand (4) and category L3+L4 names (6); brand, shop and 4
  category id embeddings → MLP → 128-d. No item-id embedding (most displayed items are cold). Popularity is **not** in
  the tower: it is added to the training score as a learned offset (`pop_logit`), so the exported `tower_score` is a
  pure content cosine and is complementary to the ranker's popularity features.
* **Query + user tower:** the query shares the term embedding. Each history event = item-tower output + event type +
  log time before the query + the query it was issued under. Attention is **candidate-aware (DIN-style)**: every
  candidate attends over the short window (< 7 d, all types), conditioned on the candidate and the query, so each
  candidate gets its own user vector. The mid/long windows are summarised by per-type counts plus the exact log size,
  and exact-match flags (same product / same top category per window) enter as raw features.
* **Loss:** listwise softmax over each training slate with target ∝ 2^label − 1, plus an auxiliary in-batch softmax on
  historical (query → item) events from the last 7 days, weighted by event type. The auxiliary loss uses a popularity
  bias correction and slate hard negatives (label-0 items only, from the model's own training users).

## 5.2 Cross-fitting (`src/jds/twotower/train.py`)

The ranker (step 6) learns on training users, so their tower scores must come from towers that never saw their slates:
five folds are used — `fold k` trains on the other four folds and scores fold k; `full` is trained on all training
users and scores validation/test. Each model trains for 2 epochs, validates twice per epoch with `eval_per_epoch`,
and keeps the best checkpoint by validation graded NDCG@10. On a T4/V100, float16 AMP is used automatically; if CUDA
runs out of memory, see the README troubleshooting section.

### Key functions

* **`model_plan`** (`train.py`) lists each model's training users and the users it scores: the cross-fitting design above.
* **`Stores.context`** (`data.py`) builds one batch of history by array indexing. History is sorted oldest to newest, so each segment is one contiguous block; the code takes the last `seg_len` positions of a block, with the newest event in the last slot and earlier positions masked as padding.
* **`CandidateAttention`** (`model.py`) scores each history event against (candidate + query). It always includes a learned null slot, so a candidate unrelated to the history can attend to nothing.
* **`QueryUserTower.forward`** (`model.py`) joins the query, the attention output, their product (an explicit query–history interaction), the per-type counts and the exact-match flags, then returns one user vector per candidate.
* **`listwise_loss`** (`train.py`) is softmax cross-entropy over a slate. The target is the graded gain normalised to sum to 1, padding is excluded, and slates with no positive are skipped.
* **`inbatch_loss`** (`train.py`) is the auxiliary sampled softmax. Each anchor's positive competes with other anchors' items and with up to 8 hard negatives from its own slate; rows holding the same item are not treated as negatives.
* **`TwoTower.logq`** (`model.py`) is the popularity correction used by `inbatch_loss`. Anchors come from the logs, so common items appear as negatives more often and would otherwise be pushed down just for being common; a learned multiple of the item's popularity buckets is subtracted from the logits. The slate loss does not need it, as its candidates are not sampled, and it uses `pop_logit` instead.
* **`word_drop`** (`train.py`) hides 15% of title terms in training (candidates and history alike), so the tower cannot rely on one distinctive word. Validation and scoring see the full title.
* **`train_one`** (`train.py`) runs the loop: AdamW with a one-cycle learning rate, an auxiliary step every 2 listwise steps, gradient clipping, and validation that keeps the best checkpoint.
* **`score_users`** (`train.py`) exports the plain cosine for every displayed item. The popularity offset and temperature are training-only, so the ranker receives a score free of explicit popularity. **`infer.score`** applies the right model to each user.

In [2]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU will be very slow)")

GPU: NVIDIA H100 80GB HBM3


In [3]:
run.stage("03_train_tower.py")

$ /home/ubuntu/bt4222-two-tower-recsys/.venv/bin/python -u scripts/03_train_tower.py --config configs/default.yaml


07:27:49 tower | [fold0] exists, skipping (use --force to retrain)


07:27:49 tower | [fold1] exists, skipping (use --force to retrain)


07:27:49 tower | [fold2] exists, skipping (use --force to retrain)


07:27:49 tower | [fold3] exists, skipping (use --force to retrain)


07:27:49 tower | [fold4] exists, skipping (use --force to retrain)


07:27:49 tower | [full] exists, skipping (use --force to retrain)


In [4]:
run.stage("04_score_tower.py")

$ /home/ubuntu/bt4222-two-tower-recsys/.venv/bin/python -u scripts/04_score_tower.py --config configs/default.yaml


07:27:51 tower | tower_scores.parquet exists (use --force to rescore)


In [5]:
banner()
import pandas as pd, json
for f in sorted(run.p.tower.glob("*_history.json")):
    print(f.stem, "(validation metrics by epoch)")
    h = pd.DataFrame(json.loads(f.read_text()))
    cols = [c for c in ["epoch", "step", "train_loss", "mrr", "ndcg10", "gndcg10"] if c in h.columns]
    display(h[cols].style.format(precision=4).highlight_max(subset=["gndcg10"], color="#c6efce"))

fold0_history (validation metrics by epoch)


,epoch,step,mrr,ndcg10,gndcg10
0,0,858,0.1918,0.1690,0.1670
1,0,1716,0.1970,0.1754,0.1732
2,0,1717,0.1971,0.1757,0.1735
3,1,858,0.2061,0.1849,0.1827
4,1,1716,0.2062,0.1850,0.1827
5,1,1717,0.2062,0.1850,0.1826


fold1_history (validation metrics by epoch)


,epoch,step,mrr,ndcg10,gndcg10
0,0,858,0.1943,0.1722,0.1697
1,0,1716,0.2001,0.1776,0.1752
2,1,858,0.2048,0.1834,0.1811
3,1,1716,0.2074,0.1858,0.1836


fold2_history (validation metrics by epoch)


,epoch,step,mrr,ndcg10,gndcg10
0,0,858,0.1946,0.1728,0.1703
1,0,1716,0.1985,0.1771,0.1747
2,0,1717,0.1980,0.1769,0.1745
3,1,858,0.2019,0.1807,0.1785
4,1,1716,0.2035,0.1819,0.1799
5,1,1717,0.2035,0.1819,0.1799


... (2 outputs left out)

fold4_history (validation metrics by epoch)


,epoch,step,mrr,ndcg10,gndcg10
0,0,858,0.1939,0.1709,0.1682
1,0,1716,0.1990,0.1757,0.1732
2,0,1717,0.1993,0.1761,0.1735
3,1,858,0.2034,0.1816,0.1790
4,1,1716,0.2060,0.1844,0.1817
5,1,1717,0.2060,0.1844,0.1817


full_history (validation metrics by epoch)


,epoch,step,mrr,ndcg10,gndcg10
0,0,1073,0.1957,0.1721,0.1700
1,0,2146,0.1984,0.1755,0.1731
2,1,1073,0.2085,0.1859,0.1837
3,1,2146,0.2062,0.1850,0.1828


In [6]:
banner()
pd.DataFrame(run.json("tower.json")).style.format(precision=4)

,model,mrr,p1,ndcg10,gndcg10,hr10,r10,r50,n_users
0,two-tower [train (out-of-fold)],0.2049,0.1224,0.1826,0.1800,0.3554,0.2703,0.7106,137382
1,two-tower [val],0.2085,0.1254,0.1859,0.1837,0.3586,0.2736,0.7195,17172
2,two-tower [test],0.2047,0.1237,0.1822,0.1797,0.3492,0.2674,0.7119,17174
